# Amu Darya — v4: **seasonal-anomaly** forecastingv3 showed that a plain seasonal climatology already reaches NSE 0.927 at this gauge, so raw-dischargeNSE mostly rewards reproducing the seasonal cycle rather than genuine predictive skill.**v4 forecasts the anomaly** `A[t] = Q[t] − clim[month(t)]`, with `clim` estimated on the trainingperiod only. Every model — including the baselines — now competes on the part of the signal thatclimatology cannot explain.Reported metrics:* **Anomaly NSE / RMSE** — the honest skill measure. `> 0` means the model beats climatology.* **Reconstructed-Q metrics** — `Q̂ = clim + Â`, for comparability with v3 and the literature.* **Skill score** `SS = 1 − MSE_model / MSE_climatology` on Q.Keeps every v2/v3 control: basin-filtered gauge, causal rolling STL, out-of-fold stacking,horizon-lagged predictors (no future weather). Horizons h = 1, 3, 6. Runtime ~35–50 min.

In [ ]:
!pip install -q cdsapi shap psutil
!pip install -q --upgrade "xarray>=2024.1" "netCDF4>=1.6"
print("ok")

In [ ]:
import os,time,json,warnings,itertools,platform
import numpy as np,pandas as pd
warnings.filterwarnings("ignore")

CDS_URL="https://cds.climate.copernicus.eu/api"
CDS_KEY="YOUR-CDS-API-KEY"
GPKG_PATH="/content/drive/MyDrive/CA-discharge.gpkg"

AMU_BBOX=[41.0,58.0,34.0,75.0]
ERA5_BUFFER_DEG=1.5
YEAR_START,YEAR_END=1979,2020
MIN_MONTHS=240
TEST_FRACTION=0.20
OOF_START_FRAC=0.55
HORIZONS=[1,3,6]

SEED=42; np.random.seed(SEED)
OUT="/content/outputs"; os.makedirs(OUT,exist_ok=True)
print("out:",OUT)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
assert os.path.exists(GPKG_PATH)
print(f"{os.path.getsize(GPKG_PATH)/1024**2:.1f} MB")

## Gauge selection (basin-filtered)

In [ ]:
import geopandas as gpd,fiona,sqlite3
DATE_H=["date","time","datetime","day","month"]
Q_H=["discharge","flow","runoff","streamflow","q_","qobs","value"]
ST_H=["station","gauge","site","code","id"]
def pick(cols,hints,ex=()):
    for h in hints:
        for c in cols:
            if h in c.lower() and c not in ex: return c
con=sqlite3.connect(GPKG_PATH)
tabs=pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'",con)["name"]
best=(None,-1,None)
for t in [x for x in tabs if not x.startswith(("gpkg_","sqlite_","rtree_"))]:
    try: cols=list(pd.read_sql(f'SELECT * FROM "{t}" LIMIT 1',con).columns)
    except: continue
    d=pick(cols,DATE_H); q=pick(cols,Q_H,(d,))
    if d and q:
        nn=pd.read_sql(f'SELECT COUNT(*) n FROM "{t}"',con)["n"][0]
        if nn>best[1]: best=(t,nn,(d,q,pick(cols,ST_H,(d,q))))
TABLE=best[0]; COL_DATE,COL_Q,COL_ST=best[2]
raw=pd.read_sql(f'SELECT * FROM "{TABLE}"',con); con.close()
raw[COL_DATE]=pd.to_datetime(raw[COL_DATE],errors="coerce")
raw=raw.dropna(subset=[COL_DATE,COL_Q])
raw=raw[(raw[COL_DATE].dt.year>=YEAR_START)&(raw[COL_DATE].dt.year<=YEAR_END)]

coords={}
for lyr in fiona.listlayers(GPKG_PATH):
    try: g=gpd.read_file(GPKG_PATH,layer=lyr)
    except: continue
    if g.geometry.isna().all() or COL_ST not in g.columns: continue
    if g.crs is not None and g.crs.to_epsg()!=4326: g=g.to_crs(4326)
    for _,r in g.iterrows():
        if r.geometry is not None:
            coords[r[COL_ST]]=(float(r.geometry.centroid.y),float(r.geometry.centroid.x))
N,W,S,E=AMU_BBOX
lens=raw.groupby(COL_ST)[COL_Q].count().sort_values(ascending=False)
inside=[s for s in lens.index if s in coords and S<=coords[s][0]<=N and W<=coords[s][1]<=E]
STATION=inside[0]; LAT,LON=coords[STATION]
df_q=(raw[raw[COL_ST]==STATION].set_index(COL_DATE)[COL_Q].astype(float)
      .resample("MS").mean().to_frame("Q").reset_index().rename(columns={COL_DATE:"date"}))
assert len(df_q)>=MIN_MONTHS
B=ERA5_BUFFER_DEG
ERA5_AREA=[round(LAT+B,2),round(LON-B,2),round(LAT-B,2),round(LON+B,2)]
print(f"Gauge {STATION} ({LAT:.3f}N,{LON:.3f}E) | {len(df_q)} months | ERA5 {ERA5_AREA}")

## ERA5-Land

In [ ]:
import cdsapi
ERA5_FILE=f"/content/era5_{STATION}.nc"
if not(os.path.exists(ERA5_FILE) and os.path.getsize(ERA5_FILE)>1000):
    cdsapi.Client(url=CDS_URL,key=CDS_KEY).retrieve("reanalysis-era5-land-monthly-means",
      {"product_type":["monthly_averaged_reanalysis"],
       "variable":["2m_temperature","total_precipitation","potential_evaporation",
                   "snow_depth_water_equivalent"],
       "year":[str(y) for y in range(YEAR_START,YEAR_END+1)],
       "month":[f"{m:02d}" for m in range(1,13)],"time":["00:00"],
       "area":ERA5_AREA,"data_format":"netcdf","download_format":"unarchived"},ERA5_FILE)
import xarray as xr
ds=xr.open_dataset(ERA5_FILE)
TIME="valid_time" if "valid_time" in ds.coords else "time"
sp=[d for d in ["latitude","longitude"] if d in ds.dims]
de=ds.mean(dim=sp).to_dataframe().reset_index().rename(columns={TIME:"date"})
rn={"t2m":"temp","tp":"precip","pev":"pet","sd":"swe"}
de=de.rename(columns={k:v for k,v in rn.items() if k in de.columns})
de=de[["date"]+[v for v in rn.values() if v in de.columns]]
de["temp"]=de["temp"]-273.15; de["precip"]=de["precip"]*1000*30.44
de["pet"]=-de["pet"]*1000*30.44; de["swe"]=de["swe"]*1000
de["date"]=pd.to_datetime(de["date"]).dt.to_period("M").dt.to_timestamp()
df_era=de.groupby("date").mean().reset_index()
print("ERA5 ok")

## Data layer + **anomaly transform**`clim[m]` is the mean discharge of calendar month *m* over the **training period only**, so notest information enters the anomaly definition. Exogenous variables are standardised intoanomalies the same way, which removes their seasonal cycle as well — otherwise a model couldrecover the seasonal signal indirectly through temperature.

In [ ]:
t0=time.time()
df=pd.merge(df_q,df_era,on="date",how="inner").sort_values("date").reset_index(drop=True)
n_missing=int(df.isna().sum().sum())
df=df.set_index("date").interpolate(method="time").bfill().ffill().reset_index()
lq=np.log1p(df.Q); n_out=int((np.abs((lq-lq.mean())/lq.std())>3).sum())

EXOG=[c for c in ["precip","temp","pet","swe"] if c in df.columns]
n=len(df); n_test=int(np.floor(n*TEST_FRACTION)); n_train=n-n_test
df["month"]=df.date.dt.month

# --- climatology from TRAIN only ---
tr=df.iloc[:n_train]
CLIM_Q={m:float(v) for m,v in tr.groupby("month").Q.mean().items()}
CLIM_X={c:{m:float(v) for m,v in tr.groupby("month")[c].mean().items()} for c in EXOG}

df["clim"]=df.month.map(CLIM_Q)
df["A"]=df.Q-df["clim"]                       # discharge anomaly = target
for c in EXOG:
    df[c+"_a"]=df[c]-df.month.map(CLIM_X[c])  # exogenous anomalies
EXOG_A=[c+"_a" for c in EXOG]
t_data=time.time()-t0

Y=df.Q.values; A=df.A.values; CLIM=df["clim"].values
EXOG_ALL=df[EXOG_A].values
IDX=pd.DatetimeIndex(df.date,freq="MS")

print(f"{n} months {df.date.min().date()}->{df.date.max().date()} | train {n_train} test {n_test}")
print(f"imputed {n_missing} outliers {n_out} | data layer {t_data*1000:.1f} ms")
print(f"Q  std {Y.std():.1f} | anomaly std {A.std():.1f} "
      f"({A.std()/Y.std()*100:.0f}% of total variability)")
print("Anomaly correlation with exogenous anomalies:",
      {c: round(np.corrcoef(A,df[c+"_a"])[0,1],3) for c in EXOG})
df.to_csv(f"{OUT}/merged_dataset.csv",index=False)

## Metrics, causal STL on the anomaly, ARIMA order

In [ ]:
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
def nse(o,s):
    o,s=np.asarray(o,float),np.asarray(s,float)
    return float(1-np.sum((o-s)**2)/np.sum((o-o.mean())**2))
def rmse(o,s):
    o,s=np.asarray(o,float),np.asarray(s,float); return float(np.sqrt(np.mean((o-s)**2)))
def kge(o,s):
    o,s=np.asarray(o,float),np.asarray(s,float)
    r=np.corrcoef(o,s)[0,1]; a=s.std()/o.std(); b=s.mean()/o.mean()
    return float(1-np.sqrt((r-1)**2+(a-1)**2+(b-1)**2))
def mae(o,s): return float(np.mean(np.abs(np.asarray(o,float)-np.asarray(s,float))))

_c={}
def decomp(t):
    if t not in _c:
        r=STL(pd.Series(A[:t],index=IDX[:t]),period=12,robust=True).fit()
        _c[t]=(r.trend.values+r.seasonal.values,r.resid.values)
    return _c[t]
def fit_arima(y,o):
    return SARIMAX(y,order=o,enforce_stationarity=False,enforce_invertibility=False).fit(disp=False)

t0=time.time(); TS_tr,R_tr=decomp(n_train); t_stl=time.time()-t0
best_aic,best_order=np.inf,(1,0,1)
t0=time.time()
for p,d_,q in itertools.product([0,1,2],[0,1],[0,1,2]):
    try:
        mm=fit_arima(TS_tr,(p,d_,q))
        if np.isfinite(mm.aic) and mm.aic<best_aic: best_aic,best_order=mm.aic,(p,d_,q)
    except: pass
t_asearch=time.time()-t0
print(f"STL {t_stl*1000:.0f}ms | ARIMA {best_order} AIC {best_aic:.1f} ({t_asearch:.1f}s)")

## LSTM + causal rolling engine (anomaly target, horizon-lagged inputs)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM,Dense,Dropout,Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler
tf.random.set_seed(SEED)
LOOKBACK,UNITS,LAYERS,DROP,EPOCHS,BATCH,LR=12,64,2,0.2,200,16,0.001

def build():
    m=Sequential([Input(shape=(LOOKBACK,1+len(EXOG_A)))])
    for i in range(LAYERS):
        m.add(LSTM(UNITS,return_sequences=(i<LAYERS-1))); m.add(Dropout(DROP))
    m.add(Dense(1)); m.compile(optimizer=tf.keras.optimizers.Adam(LR),loss="mse")
    return m
def train_lstm(t_s,e_s,hi,h):
    X,y=[],[]
    for i in range(LOOKBACK+h-1,hi):
        e=i-h+1; X.append(np.column_stack([t_s[e-LOOKBACK:e],e_s[e-LOOKBACK:e]])); y.append(t_s[i])
    X,y=np.array(X),np.array(y); m=build()
    hist=m.fit(X,y,epochs=EPOCHS,batch_size=BATCH,validation_split=0.15,
      callbacks=[EarlyStopping(monitor="val_loss",patience=20,restore_best_weights=True)],verbose=0)
    return m,hist
def lstm_step(m,t_s,e_s,t,h,sc):
    e=t-h+1; blk=np.column_stack([t_s[e-LOOKBACK:e],e_s[e-LOOKBACK:e]])[None,...]
    return float(sc.inverse_transform([[m.predict(blk,verbose=0).ravel()[0]]])[0,0])
def rolling(start,stop,h,lm,sc_r,sc_x,label=""):
    a,l=[],[]; ta=tl=0.0; e_s=sc_x.transform(EXOG_ALL)
    for t in range(start,stop):
        org=t-h+1; TS_t,R_t=decomp(org)
        s=time.time(); a.append(float(fit_arima(TS_t,best_order).forecast(h)[-1])); ta+=time.time()-s
        R_s=sc_r.transform(R_t.reshape(-1,1)).ravel()
        R_full=np.concatenate([R_s,np.zeros(len(A)-len(R_s))])
        s=time.time(); l.append(lstm_step(lm,R_full,e_s,t,h,sc_r)); tl+=time.time()-s
    if label: print(f"  {label} h={h}: {stop-start} steps, ARIMA {ta:.0f}s")
    return np.array(a),np.array(l),ta,tl
def wf_arima(series,start,stop,h,o):
    return np.array([float(fit_arima(series[:t-h+1],o).forecast(h)[-1]) for t in range(start,stop)])
print("ready")

## Run all horizonsEvery model predicts the **anomaly**; discharge is then reconstructed as `Q̂ = clim + Â`.The climatology baseline is simply `Â = 0`.

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit,RandomizedSearchCV
PARAMS={"n_estimators":[200,400,600,800],"max_depth":[2,3,4,5,6],
        "learning_rate":[0.01,0.03,0.05,0.1],"subsample":[0.7,0.8,0.9,1.0],
        "colsample_bytree":[0.7,0.8,0.9,1.0],"reg_lambda":[0.5,1.0,2.0,5.0],
        "gamma":[0,0.1,0.5,1.0]}
oof_start=int(n_train*OOF_START_FRAC)
obs_A=A[n_train:]; obs_Q=Y[n_train:]; clim_te=CLIM[n_train:]
test_dates=pd.to_datetime(df.date.values[n_train:])
mse_clim=float(np.mean((obs_Q-clim_te)**2))
RES={}; TIMING={}
print(f"OOF {oof_start}->{n_train} | test {n_train}->{n}")
print(f"Climatology on test: NSE(Q) {nse(obs_Q,clim_te):.3f}  RMSE {np.sqrt(mse_clim):.2f}")

def score(nm,h,a_hat):
    q_hat=clim_te+a_hat
    return {"Horizon (months)":h,"Model":nm,
            "NSE (anomaly)":round(nse(obs_A,a_hat),3),
            "RMSE (anomaly)":round(rmse(obs_A,a_hat),2),
            "NSE (Q)":round(nse(obs_Q,q_hat),3),
            "KGE (Q)":round(kge(obs_Q,q_hat),3),
            "RMSE (Q)":round(rmse(obs_Q,q_hat),2),
            "MAE (Q)":round(mae(obs_Q,q_hat),2),
            "Skill vs clim.":round(1-np.mean((obs_Q-q_hat)**2)/mse_clim,3)}

for h in HORIZONS:
    print(f"\n===== h={h} =====")
    FEAT=["ARIMA_pred","LSTM_resid_pred",f"A_lag{h}",f"A_lag{h+11}"]+[f"{c}_lag{h}" for c in EXOG]
    # OOF
    TS_o,R_o=decomp(oof_start)
    scr_o=MinMaxScaler().fit(R_o.reshape(-1,1)); scx_o=MinMaxScaler().fit(EXOG_ALL[:oof_start])
    Rs_o=np.concatenate([scr_o.transform(R_o.reshape(-1,1)).ravel(),np.zeros(len(A)-oof_start)])
    lo,_=train_lstm(Rs_o,scx_o.transform(EXOG_ALL),oof_start,h)
    a_o,l_o,_,_=rolling(oof_start,n_train,h,lo,scr_o,scx_o,"OOF")
    # test
    scr=MinMaxScaler().fit(R_tr.reshape(-1,1)); scx=MinMaxScaler().fit(EXOG_ALL[:n_train])
    Rs=np.concatenate([scr.transform(R_tr.reshape(-1,1)).ravel(),np.zeros(len(A)-n_train)])
    t0=time.time(); lm,hist=train_lstm(Rs,scx.transform(EXOG_ALL),n_train,h); t_ltr=time.time()-t0
    a_t,l_t,t_ar,t_li=rolling(n_train,n,h,lm,scr,scx,"TEST")
    # meta
    def mk(a,l,lo_,hi_):
        return np.column_stack([a,l,A[lo_-h:hi_-h],A[lo_-h-11:hi_-h-11],EXOG_ALL[lo_-h:hi_-h]])
    M_tr=mk(a_o,l_o,oof_start,n_train); y_tr=A[oof_start:n_train]; M_te=mk(a_t,l_t,n_train,n)
    t0=time.time()
    sr=RandomizedSearchCV(XGBRegressor(random_state=SEED,objective="reg:squarederror"),
      PARAMS,n_iter=40,cv=TimeSeriesSplit(n_splits=5),scoring="neg_root_mean_squared_error",
      random_state=SEED,n_jobs=-1).fit(M_tr,y_tr)
    t_xtr=time.time()-t0; meta=sr.best_estimator_
    t0=time.time(); ens=meta.predict(M_te); t_xin=time.time()-t0
    # baselines on the anomaly
    ar_raw=wf_arima(A,n_train,n,h,best_order)
    sca=MinMaxScaler().fit(A[:n_train].reshape(-1,1)); As=sca.transform(A.reshape(-1,1)).ravel()
    la_,_=train_lstm(As,scx.transform(EXOG_ALL),n_train,h)
    ls_raw=np.array([lstm_step(la_,As,scx.transform(EXOG_ALL),t,h,sca) for t in range(n_train,n)])
    dfx=df.copy()
    for lg in [h,h+1,h+2,h+11]: dfx[f"Al{lg}"]=dfx.A.shift(lg)
    for c in EXOG_A: dfx[f"{c}l{h}"]=dfx[c].shift(h)
    ft=[f"Al{lg}" for lg in [h,h+1,h+2,h+11]]+[f"{c}l{h}" for c in EXOG_A]
    dfx=dfx.dropna(subset=ft).reset_index(drop=True); ntx=len(dfx)-n_test
    sx=RandomizedSearchCV(XGBRegressor(random_state=SEED,objective="reg:squarederror"),
      PARAMS,n_iter=40,cv=TimeSeriesSplit(n_splits=5),scoring="neg_root_mean_squared_error",
      random_state=SEED,n_jobs=-1).fit(dfx[ft].values[:ntx],dfx.A.values[:ntx])
    xg_raw=sx.best_estimator_.predict(dfx[ft].values[ntx:])

    tab=pd.DataFrame([score("Climatology",h,np.zeros_like(obs_A)),
        score("ARIMA",h,ar_raw),score("LSTM",h,ls_raw),score("XGBoost",h,xg_raw),
        score("STL-ARIMA",h,a_t),score("STL-ARIMA-LSTM",h,a_t+l_t),
        score("Proposed (STL-ARIMA-LSTM-XGBoost)",h,ens)])
    print(tab.to_string(index=False))
    RES[h]=dict(tab=tab,ens=ens,ar=ar_raw,ls=ls_raw,xg=xg_raw,sa=a_t,sal=a_t+l_t,
                meta=meta,M_te=M_te,FEAT=FEAT,best=sr.best_params_,epochs=len(hist.history["loss"]))
    TIMING[h]=dict(t_ltr=t_ltr,t_ar=t_ar,t_li=t_li/n_test,t_xtr=t_xtr,t_xin=t_xin)

## Table III + verdict

In [ ]:
tab3=pd.concat([RES[h]["tab"] for h in HORIZONS],ignore_index=True)
tab3.to_csv(f"{OUT}/table3_model_performance.csv",index=False)
print(tab3.to_string(index=False)); print()
for h in HORIZONS:
    t=RES[h]["tab"]; p=t[t.Model.str.startswith("Proposed")].iloc[0]
    riv=t[~t.Model.str.startswith("Proposed")]; b=riv.loc[riv["NSE (anomaly)"].idxmax()]
    beats_clim = p["Skill vs clim."]>0
    print(f"h={h}: Proposed anomaly-NSE {p['NSE (anomaly)']}, skill vs clim {p['Skill vs clim.']} "
          f"({'BEATS' if beats_clim else 'does NOT beat'} climatology) | "
          f"best rival {b.Model} {b['NSE (anomaly)']} -> "
          f"{'WINS' if p['NSE (anomaly)']>b['NSE (anomaly)'] else 'LOSES'}")

## Table IV — Diebold-Mariano (on the anomaly)

In [ ]:
from scipy import stats
def dm(o,p1,p2,h=1):
    e1=np.asarray(o,float)-np.asarray(p1,float); e2=np.asarray(o,float)-np.asarray(p2,float)
    d=e1**2-e2**2; m=len(d); db=d.mean(); g0=np.sum((d-db)**2)/m
    gs=sum(2*np.sum((d[k:]-db)*(d[:-k]-db))/m for k in range(1,h))
    s=db/np.sqrt((g0+gs)/m); return float(s),float(2*(1-stats.norm.cdf(abs(s))))
rows=[]
for h in HORIZONS:
    R=RES[h]
    for lbl,o in [("Climatology",np.zeros_like(obs_A)),("ARIMA",R["ar"]),("LSTM",R["ls"]),
                  ("XGBoost",R["xg"]),("STL-ARIMA",R["sa"]),("STL-ARIMA-LSTM",R["sal"])]:
        s,p=dm(obs_A,R["ens"],o,h=h)
        rows.append({"Horizon (months)":h,"Comparison":f"Proposed vs. {lbl}",
                     "DM statistic":round(s,3),"p-value":round(p,4),
                     "Significant (a=0.05)":"Yes" if p<0.05 else "No"})
tab4=pd.DataFrame(rows); tab4.to_csv(f"{OUT}/table4_dm_test.csv",index=False)
print(tab4.to_string(index=False))
print("\nNegative DM = Proposed has the lower squared error.")

## Fig. 5 — SHAP and regime contribution

In [ ]:
import shap,matplotlib
matplotlib.rcParams["font.family"]="serif"
import matplotlib.pyplot as plt
sa=[];ra=[]
for h in HORIZONS:
    R=RES[h]; sv=shap.TreeExplainer(R["meta"]).shap_values(R["M_te"])
    s=pd.DataFrame({"Horizon (months)":h,"Feature":R["FEAT"],
                    "Mean |SHAP|":np.abs(sv).mean(axis=0).round(3)})
    s["Relative (%)"]=(s["Mean |SHAP|"]/s["Mean |SHAP|"].sum()*100).round(1)
    sa.append(s.sort_values("Mean |SHAP|",ascending=False))
    q70,q30=np.percentile(obs_Q,70),np.percentile(obs_Q,30)
    for nm,mk in {"High (>P70)":obs_Q>q70,"Mid":(obs_Q>=q30)&(obs_Q<=q70),
                  "Low (<P30)":obs_Q<q30}.items():
        if mk.sum()==0: continue
        v=np.abs(sv[mk]).mean(axis=0); sh=v/v.sum()*100
        ra.append({"Horizon (months)":h,"Flow regime":nm,"n":int(mk.sum()),
          "ARIMA (%)":round(sh[0],1),"LSTM (%)":round(sh[1],1),
          "Lagged anomaly (%)":round(sh[2:4].sum(),1),"Exogenous (%)":round(sh[4:].sum(),1),
          "RMSE (anomaly)":round(rmse(obs_A[mk],R["ens"][mk]),2)})
    if h==HORIZONS[0]:
        plt.figure(figsize=(6.5,4))
        shap.summary_plot(sv,R["M_te"],feature_names=R["FEAT"],show=False)
        plt.tight_layout(); plt.savefig(f"{OUT}/fig5_shap.png",dpi=300,bbox_inches="tight"); plt.show()
pd.concat(sa,ignore_index=True).to_csv(f"{OUT}/fig5_shap_data.csv",index=False)
pd.DataFrame(ra).to_csv(f"{OUT}/regime_contribution.csv",index=False)
print(pd.concat(sa,ignore_index=True).to_string(index=False)); print()
print(pd.DataFrame(ra).to_string(index=False))

## Tables II & V, metadata, export

In [ ]:
import psutil,sklearn,xgboost,statsmodels
h0=HORIZONS[0]; T=TIMING[h0]
mem=psutil.Process(os.getpid()).memory_info().rss/1024**2
t0=time.time(); _=RES[h0]["meta"].predict(RES[h0]["M_te"][-1:]); t_e2e=time.time()-t0
tab5=pd.DataFrame([
 ["Perception + data layer","Ingestion, cleaning, anomaly transform",f"{t_data*1000:.1f} ms"],
 ["Model layer - STL","Causal decomposition",f"{t_stl*1000:.1f} ms"],
 ["Model layer - ARIMA","Order selection (offline)",f"{t_asearch:.1f} s"],
 ["Model layer - ARIMA",f"Walk-forward, full test (h={h0})",f"{T['t_ar']:.1f} s"],
 ["Model layer - LSTM","Training (offline)",f"{T['t_ltr']:.1f} s"],
 ["Model layer - LSTM","Inference (per step)",f"{T['t_li']*1000:.1f} ms"],
 ["Model layer - XGBoost","Training + tuning (offline)",f"{T['t_xtr']:.1f} s"],
 ["Model layer - XGBoost","Inference (full test)",f"{T['t_xin']*1000:.2f} ms"],
 ["Application layer","Single-step forecast latency",f"{t_e2e*1000:.2f} ms"],
 ["System","Peak memory footprint",f"{mem:.0f} MB"]],columns=["Component","Metric","Value"])
tab5.to_csv(f"{OUT}/table5_system_performance.csv",index=False)
print(tab5.to_string(index=False))

rows=[{"Component":"Anomaly transform","Parameter":"climatology","Value":"monthly means",
       "Selection":"training period only"},
 {"Component":"STL","Parameter":"period","Value":12,"Selection":"annual cycle"},
 {"Component":"STL","Parameter":"refit scheme","Value":"causal rolling","Selection":"past only"},
 {"Component":"ARIMA","Parameter":"order (p,d,q)","Value":str(best_order),"Selection":"grid, AIC"},
 {"Component":"LSTM","Parameter":"lookback","Value":LOOKBACK,"Selection":"fixed"},
 {"Component":"LSTM","Parameter":"layers","Value":LAYERS,"Selection":"fixed"},
 {"Component":"LSTM","Parameter":"units","Value":UNITS,"Selection":"fixed"},
 {"Component":"LSTM","Parameter":"dropout","Value":DROP,"Selection":"fixed"},
 {"Component":"LSTM","Parameter":"optimizer / lr","Value":f"Adam / {LR}","Selection":"fixed"},
 {"Component":"LSTM","Parameter":"batch","Value":BATCH,"Selection":"fixed"},
 {"Component":"Meta-learner","Parameter":"training data","Value":"out-of-fold",
  "Selection":f"rolling origin, {n_train-oof_start} rows"}]
for h in HORIZONS:
    for k,v in RES[h]["best"].items():
        rows.append({"Component":f"XGBoost (h={h})","Parameter":k,"Value":v,
                     "Selection":"randomized search, TimeSeriesSplit(5)"})
pd.DataFrame(rows).to_csv(f"{OUT}/table2_hyperparameters.csv",index=False)

json.dump({"station":str(STATION),"station_coordinates":{"lat":LAT,"lon":LON},
 "station_inside_basin":True,"basin_envelope_NWSE":AMU_BBOX,"era5_window_NWSE":ERA5_AREA,
 "study_period":f"{df.date.min().date()} to {df.date.max().date()}",
 "n_months_total":int(n),"n_train":int(n_train),"n_test":int(n_test),
 "oof_rows":int(n_train-oof_start),"horizons":HORIZONS,"target":"seasonal anomaly",
 "climatology_train_only":True,"climatology_test_NSE_Q":round(nse(obs_Q,clim_te),3),
 "values_imputed":n_missing,"outliers_flagged":n_out,"exogenous_features":EXOG,"seed":SEED,
 "leakage_controls":["climatology from training period only",
   "STL refitted on past data only","meta-learner trained out-of-fold",
   "scalers fitted on training data only",
   "exogenous and lagged target shifted by the forecast horizon"],
 "software":{"python":platform.python_version(),"numpy":np.__version__,"pandas":pd.__version__,
   "statsmodels":statsmodels.__version__,"scikit-learn":sklearn.__version__,
   "xgboost":xgboost.__version__,"tensorflow":tf.__version__,"shap":shap.__version__},
 "hardware":"Google Colab"},open(f"{OUT}/run_metadata.json","w"),indent=2,default=str)

pd.DataFrame({"date":test_dates,"observed_Q":obs_Q,"climatology":clim_te,"observed_anomaly":obs_A,
  **{f"predicted_Q_h{h}":clim_te+RES[h]["ens"] for h in HORIZONS},
  **{f"predicted_anomaly_h{h}":RES[h]["ens"] for h in HORIZONS}}
 ).to_csv(f"{OUT}/fig3_hydrograph_data.csv",index=False)
tab3.to_csv(f"{OUT}/fig4_comparison_data.csv",index=False)
allp={"date":test_dates,"observed_Q":obs_Q,"observed_anomaly":obs_A,"climatology":clim_te}
for h in HORIZONS:
    R=RES[h]
    for k,v in [("ARIMA",R["ar"]),("LSTM",R["ls"]),("XGBoost",R["xg"]),("STL_ARIMA",R["sa"]),
                ("STL_ARIMA_LSTM",R["sal"]),("Proposed",R["ens"])]:
        allp[f"{k}_anom_h{h}"]=v
pd.DataFrame(allp).to_csv(f"{OUT}/predictions.csv",index=False)
for f in sorted(os.listdir(OUT)): print(" ",f)

In [ ]:
!cd /content && zip -qr outputs.zip outputs
from google.colab import files; files.download("/content/outputs.zip")

In [ ]:
fig,ax=plt.subplots(len(HORIZONS),1,figsize=(11,3.2*len(HORIZONS)),sharex=True)
for i,h in enumerate(HORIZONS):
    R=RES[h]
    ax[i].axhline(0,color="grey",lw=.8)
    ax[i].plot(test_dates,obs_A,"k-",lw=1.5,label="Observed anomaly")
    ax[i].plot(test_dates,R["ens"],"r--",lw=1.2,label="Proposed")
    ax[i].plot(test_dates,R["xg"],"b:",lw=1.0,label="XGBoost baseline")
    ax[i].set_title(f"h={h} | anomaly NSE {nse(obs_A,R['ens']):.3f} | "
                    f"skill vs clim {1-np.mean((obs_Q-(clim_te+R['ens']))**2)/mse_clim:.3f}",fontsize=10)
    ax[i].legend(fontsize=8); ax[i].grid(alpha=.3); ax[i].set_ylabel("Anomaly (m$^3$/s)")
plt.tight_layout(); plt.show()